# 03a — Feature engineering : statistiques glissantes (FD001)

**Objectif** : construire des features tabulaires à partir des données préparées, en résumant les 30 derniers cycles de chaque capteur (moyenne, écart-type, min, max).

In [1]:
# --- Imports et configuration ---
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parents[1]
sys.path.append(str(PROJECT_ROOT))

from src.data.loaders import SENSOR_COLUMNS

SEED = 42
np.random.seed(SEED)

SUBSET = "FD001"
WINDOW = 30

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "ml"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
df_train = pd.read_parquet(INTERIM_DIR / f"train_{SUBSET}_prepared.parquet")
df_test = pd.read_parquet(INTERIM_DIR / f"test_{SUBSET}_prepared.parquet")
df_rul_test = pd.read_csv(INTERIM_DIR / f"rul_test_{SUBSET}_prepared.csv")

capteurs = [c for c in SENSOR_COLUMNS if c in df_train.columns]
print(f"{len(capteurs)} capteurs disponibles")
print(f"train : {df_train.shape}, test : {df_test.shape}")

15 capteurs disponibles
train : (20631, 20), test : (13096, 19)


## 1. Construction des statistiques glissantes

Pour chaque capteur, on calcule sur les 30 derniers cycles connus : la moyenne, l'écart-type, le minimum et le maximum. Pour les tout premiers cycles d'un moteur (moins de 30 disponibles), on utilise tout ce qui est disponible plutôt que d'attendre 30 cycles complets.

In [3]:
def features_glissantes(df, colonnes, window=WINDOW):
    df = df.sort_values(["unit_number", "time_in_cycles"]).reset_index(drop=True)
    groupes = df.groupby("unit_number")[colonnes]

    moyenne = groupes.rolling(window=window, min_periods=1).mean().reset_index(level=0, drop=True)
    ecart_type = groupes.rolling(window=window, min_periods=1).std().reset_index(level=0, drop=True)
    minimum = groupes.rolling(window=window, min_periods=1).min().reset_index(level=0, drop=True)
    maximum = groupes.rolling(window=window, min_periods=1).max().reset_index(level=0, drop=True)

    moyenne.columns = [f"{c}_moyenne" for c in colonnes]
    ecart_type.columns = [f"{c}_ecart_type" for c in colonnes]
    minimum.columns = [f"{c}_min" for c in colonnes]
    maximum.columns = [f"{c}_max" for c in colonnes]

    # Ecart-type indefini sur une fenetre d'un seul point -> aucune variation observee
    ecart_type = ecart_type.fillna(0)

    autres_colonnes = [c for c in df.columns if c not in colonnes]
    return pd.concat([df[autres_colonnes], moyenne, ecart_type, minimum, maximum], axis=1)


features_train = features_glissantes(df_train, capteurs)

print(features_train.shape)
features_train.head()

(20631, 65)


,unit_number,time_in_cycles,op_setting_1,op_setting_2,RUL,sensor_2_moyenne,sensor_3_moyenne,sensor_4_moyenne,sensor_6_moyenne,sensor_7_moyenne,...,sensor_8_max,sensor_9_max,sensor_11_max,sensor_12_max,sensor_13_max,sensor_14_max,sensor_15_max,sensor_17_max,sensor_20_max,sensor_21_max
0,1,1,-0.0007,-0.0004,125,0.183735,0.406802,0.309757,1.0,0.726248,...,0.242424,0.109755,0.369048,0.633262,0.205882,0.199608,0.363986,0.333333,0.713178,0.724662
1,1,2,0.0019,-0.0003,125,0.233434,0.429911,0.331195,1.0,0.677134,...,0.242424,0.109755,0.380952,0.765458,0.279412,0.199608,0.411312,0.333333,0.713178,0.731014
2,1,3,-0.0043,0.0003,125,0.270080,0.409781,0.344306,1.0,0.688137,...,0.272727,0.140043,0.380952,0.795309,0.279412,0.199608,0.411312,0.333333,0.713178,0.731014
3,1,4,0.0007,0.0000,125,0.288404,0.371376,0.341028,1.0,0.701288,...,0.318182,0.140043,0.380952,0.889126,0.294118,0.199608,0.411312,0.333333,0.713178,0.731014
4,1,5,-0.0019,-0.0002,125,0.300602,0.348594,0.353747,1.0,0.694686,...,0.318182,0.149960,0.380952,0.889126,0.294118,0.199608,0.411312,0.416667,0.713178,0.731014


**Interprétation** : chaque ligne de `features_train` correspond toujours à un cycle précis d'un moteur, mais résume maintenant ses 30 derniers cycles au lieu d'une seule mesure instantanée. Le nombre de lignes ne change pas (20631, comme avant), seul le nombre de colonnes augmente : 4 statistiques × 15 capteurs = 60 nouvelles colonnes, plus les identifiants et la cible.

## 2. Application sur le jeu de test

On calcule les mêmes statistiques sur tout l'historique de chaque moteur test, mais on ne garde que la ligne du dernier cycle enregistré — c'est le seul point pour lequel on a une RUL de référence (`RUL_FD001.txt`).

In [4]:
features_test_toutes = features_glissantes(df_test, capteurs)

dernier_cycle = features_test_toutes.groupby("unit_number")["time_in_cycles"].idxmax()
features_test = features_test_toutes.loc[dernier_cycle].reset_index(drop=True)

features_test = features_test.merge(df_rul_test[["unit_number", "RUL_finale"]], on="unit_number")
features_test = features_test.rename(columns={"RUL_finale": "RUL"})
features_test = features_test[features_train.columns]

print(features_test.shape)
features_test.head()

(100, 65)


,unit_number,time_in_cycles,op_setting_1,op_setting_2,RUL,sensor_2_moyenne,sensor_3_moyenne,sensor_4_moyenne,sensor_6_moyenne,sensor_7_moyenne,...,sensor_8_max,sensor_9_max,sensor_11_max,sensor_12_max,sensor_13_max,sensor_14_max,sensor_15_max,sensor_17_max,sensor_20_max,sensor_21_max
0,1,31,-0.0006,0.0004,112,0.327008,0.318720,0.327256,1.0,0.692915,...,0.318182,0.158081,0.386905,0.788913,0.352941,0.204768,0.510966,0.416667,0.751938,0.777410
1,2,49,0.0018,-0.0001,98,0.444880,0.397907,0.411569,1.0,0.600429,...,0.424242,0.153325,0.488095,0.754797,0.411765,0.191609,0.645633,0.583333,0.689922,0.675780
2,3,126,-0.0016,0.0004,69,0.504217,0.459676,0.512312,1.0,0.504133,...,0.469697,0.169703,0.613095,0.650320,0.485294,0.175250,0.665641,0.583333,0.720930,0.637531
3,4,106,0.0012,0.0004,82,0.459538,0.442475,0.463150,1.0,0.564895,...,0.424242,0.180876,0.505952,0.680171,0.455882,0.195634,0.639092,0.583333,0.651163,0.666529
4,5,98,-0.0013,-0.0004,91,0.431526,0.425957,0.452178,1.0,0.563231,...,0.439394,0.168536,0.541667,0.742004,0.411765,0.201466,0.626780,0.583333,0.651163,0.714582


**Interprétation** : `features_test` ne contient qu'une ligne par moteur (100 au total) — celle du dernier cycle connu, la seule pour laquelle on a une vraie valeur de RUL à comparer.

## 3. Sauvegarde

Les tables de features sont sauvegardées dans `data/processed/ml/`.

In [5]:
features_train.to_parquet(PROCESSED_DIR / f"train_{SUBSET}_features.parquet", index=False)
features_test.to_parquet(PROCESSED_DIR / f"test_{SUBSET}_features.parquet", index=False)

print("Fichiers sauvegardes dans", PROCESSED_DIR)
for f in sorted(PROCESSED_DIR.glob(f"*{SUBSET}*")):
    print(" -", f.name)

Fichiers sauvegardes dans C:\cmapss-prediction-rul\data\processed\ml
 - test_FD001_features.parquet
 - train_FD001_features.parquet


## Synthèse

Cette étape a produit une table de features par cycle (train) et une table d'une ligne par moteur (test, au dernier cycle connu), chacune avec 4 statistiques par capteur retenu sur une fenêtre de 30 cycles.